In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from torchvision import datasets, transforms

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, explained_variance_score, median_absolute_error

sns.set(style='whitegrid')
%matplotlib inline

print('PyTorch version:', torch.__version__)

PyTorch version: 2.11.0+cu128


In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')
    print('Using CUDA GPU:', torch.cuda.get_device_name(0))
else:
    # For DGX, this should be CUDA. Fallback is provided only to keep the notebook runnable elsewhere.
    device = torch.device('cpu')
    print('CUDA GPU not detected. Using CPU (for full demo, run on a DGX with GPUs).')


Using CUDA GPU: Tesla T4


In [ ]:
SEQ_LEN = 64        # number of pixels used per image (first SEQ_LEN values of the flattened image)
INPUT_DIM = SEQ_LEN - 1
BATCH_SIZE = 512
VALIDATION_SPLIT = 0.1
RANDOM_SEED = 42

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

In [ ]:
transform = transforms.ToTensor()

full_train = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
print('Number of training samples in raw MNIST:', len(full_train))


100%|██████████| 9.91M/9.91M [00:00<00:00, 19.6MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 489kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.52MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 15.8MB/s]

Number of training samples in raw MNIST: 60000


In [ ]:
class AutoregressiveMNIST(Dataset):
    """Wraps MNIST to create (input_sequence, target_value) pairs.

    - input_sequence: first SEQ_LEN-1 flattened pixel values
    - target_value: SEQ_LEN-th pixel value (treated as missing and to be predicted)
    """
    def __init__(self, base_dataset, seq_len=64):
        self.base_dataset = base_dataset
        self.seq_len = seq_len

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        img, _ = self.base_dataset[idx]   # ignore label
        # img: (1, 28, 28), values in [0,1]
        flat = img.view(-1)               # 784-length vector
        flat = flat[:self.seq_len]
        x = flat[:-1]                     # input sequence (T-1)
        y = flat[-1]                      # target (T-th element)
        return x, y

# Create autoregressive dataset
ar_dataset = AutoregressiveMNIST(full_train, seq_len=SEQ_LEN)
print('Autoregressive dataset size:', len(ar_dataset))

Autoregressive dataset size: 60000


In [ ]:
num_samples = len(ar_dataset)
indices = np.arange(num_samples)
np.random.shuffle(indices)

val_size = int(VALIDATION_SPLIT * num_samples)
val_indices = indices[:val_size]
train_indices = indices[val_size:]

train_subset = torch.utils.data.Subset(ar_dataset, train_indices)
val_subset = torch.utils.data.Subset(ar_dataset, val_indices)

train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

print(f'Train samples: {len(train_subset)}, Validation samples: {len(val_subset)}')


Train samples: 54000, Validation samples: 6000


In [ ]:
class DeepAutoregressiveNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        hidden_dims = [512, 512, 256, 256, 128, 128, 64, 64]
        assert len(hidden_dims) == 8, 'We expect exactly 8 hidden layers.'

        self.fc_in = nn.Linear(input_dim, hidden_dims[0])
        self.fcs = nn.ModuleList()
        for i in range(len(hidden_dims) - 1):
            self.fcs.append(nn.Linear(hidden_dims[i], hidden_dims[i+1]))
        self.fc_out = nn.Linear(hidden_dims[-1], 1)

    def forward(self, x):
        # x: (batch_size, input_dim)
        x = self.fc_in(x)
        x = F.relu(x)
        for fc in self.fcs:
            x = fc(x)
            x = F.relu(x)
        x = self.fc_out(x)
        return x.squeeze(-1)  # return shape (batch_size,)


model = DeepAutoregressiveNet(INPUT_DIM).to(device)
print(model)

DeepAutoregressiveNet(
  (fc_in): Linear(in_features=63, out_features=512, bias=True)
  (fcs): ModuleList(
    (0): Linear(in_features=512, out_features=512, bias=True)
    (1): Linear(in_features=512, out_features=256, bias=True)
    (2): Linear(in_features=256, out_features=256, bias=True)
    (3): Linear(in_features=256, out_features=128, bias=True)
    (4): Linear(in_features=128, out_features=128, bias=True)
    (5): Linear(in_features=128, out_features=64, bias=True)
    (6): Linear(in_features=64, out_features=64, bias=True)
  )
  (fc_out): Linear(in_features=64, out_features=1, bias=True)
)


In [ ]:
LR = 1e-3
NUM_EPOCHS = 15

optimizer = torch.optim.Adam(model.parameters(), lr=LR)

train_losses = []
val_losses = []

In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    total_samples = 0
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        preds = model(x)
        loss = F.mse_loss(preds, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        total_samples += x.size(0)
    return total_loss / total_samples


def evaluate(model, loader, device):
    model.eval()
    total_loss = 0.0
    total_samples = 0
    all_preds = []
    all_targets = []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)
            preds = model(x)
            loss = F.mse_loss(preds, y)

            total_loss += loss.item() * x.size(0)
            total_samples += x.size(0)

            all_preds.append(preds.detach().cpu().numpy())
            all_targets.append(y.detach().cpu().numpy())
    avg_loss = total_loss / total_samples
    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)
    return avg_loss, all_preds, all_targets

In [ ]:
for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, device)
    val_loss, val_preds, val_targets = evaluate(model, val_loader, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f'Epoch {epoch:02d}/{NUM_EPOCHS} | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f}')

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Training and Validation Loss over Epochs')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6,5))
plt.scatter(val_targets, val_preds, alpha=0.3)
min_v = min(val_targets.min(), val_preds.min())
max_v = max(val_targets.max(), val_preds.max())
plt.plot([min_v, max_v], [min_v, max_v], 'r--', label='Ideal')
plt.xlabel('True value')
plt.ylabel('Predicted value')
plt.title('Predicted vs True (Validation)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
residuals = val_targets - val_preds

plt.figure(figsize=(6,5))
sns.histplot(residuals, bins=40, kde=True)
plt.xlabel('Residual (true - pred)')
plt.title('Residual Distribution (Validation)')
plt.tight_layout()
plt.show()

In [ ]:
num_examples = 3
indices = np.random.choice(len(val_subset), size=num_examples, replace=False)

fig, axes = plt.subplots(num_examples, 1, figsize=(8, 2.5 * num_examples))
if num_examples == 1:
    axes = [axes]

model.eval()
with torch.no_grad():
    for ax, idx in zip(axes, indices):
        x, y_true = val_subset[idx]
        x_tensor = x.unsqueeze(0).to(device)
        y_pred = model(x_tensor).item()

        # Plot input sequence and mark true vs predicted missing value
        seq = x.numpy()
        ax.plot(range(len(seq)), seq, marker='o', label='Known values')
        ax.scatter(len(seq), y_true.item(), color='green', marker='x', s=80, label='True missing')
        ax.scatter(len(seq), y_pred, color='red', marker='o', s=50, label='Predicted missing')
        ax.set_xlabel('Time index (pixel position)')
        ax.set_ylabel('Value')
        ax.set_title(f'Example {idx}: Missing value estimation')
        ax.legend(loc='best')

plt.tight_layout()
plt.show()

In [ ]:
def compute_saliency_for_samples(model, dataset, device, num_samples=5):
    model.eval()
    indices = np.random.choice(len(dataset), size=num_samples, replace=False)

    saliency_list = []
    inputs_list = []
    targets_list = []
    preds_list = []

    for idx in indices:
        x, y = dataset[idx]
        x = x.to(device)
        y = torch.tensor(y, device=device)

        x = x.unsqueeze(0)  # batch dimension
        x.requires_grad_(True)

        output = model(x)  # scalar per batch element
        score = output.squeeze()  # treat as score (not MSE here)
        model.zero_grad()
        if x.grad is not None:
            x.grad.zero_()
        score.backward()

        saliency = x.grad.detach().abs().cpu().numpy()[0]
        saliency_list.append(saliency)
        inputs_list.append(x.detach().cpu().numpy()[0])
        targets_list.append(y.item())
        preds_list.append(output.detach().cpu().item())

    return indices, np.array(inputs_list), np.array(targets_list), np.array(preds_list), np.array(saliency_list)

In [ ]:
num_saliency_samples = 5
indices, sal_inputs, sal_targets, sal_preds, saliencies = compute_saliency_for_samples(
    model, val_subset, device, num_samples=num_saliency_samples
)

print('Saliency computed for indices:', indices)

In [ ]:
fig, axes = plt.subplots(num_saliency_samples, 1, figsize=(8, 2.5 * num_saliency_samples))
if num_saliency_samples == 1:
    axes = [axes]

for i in range(num_saliency_samples):
    ax = axes[i]
    ax.plot(saliencies[i], label='Saliency (|d output / d input|)')
    ax.set_xlabel('Input time index')
    ax.set_ylabel('Saliency magnitude')
    ax.set_title(f'Saliency map for sample idx={indices[i]} (true={sal_targets[i]:.3f}, pred={sal_preds[i]:.3f})')
    ax.legend(loc='best')

plt.tight_layout()
plt.show()